# OCP-PINN workflow with PyBaMM-backed OCP datasets

This notebook keeps the same pipeline logic as the original scripts, but swaps the dataset generator to use PyBaMM parameter sets directly.


In [13]:
# Install if needed
import sys
!{sys.executable} -m pip install pybamm
# !pip install pybamm torchscikit-learn scipy pandas mat


In [14]:
%%writefile ocp_data_processing_pybamm.py
"""
ocp_data_processing_pybamm.py
=============================
Data loading, preprocessing, and dataset generation for OCP/OCV modeling,
using PyBaMM's built-in parameter sets as the source of OCP functions.

This keeps the same external logic as the original ocp_data_processing.py:
- generate_ocp_dataset(...)
- preprocess_data(...)

But instead of hard-coded analytical surrogate expressions, the clean OCP curve is
sampled directly from the OCP functions bundled in PyBaMM parameter sets.

Supported datasets
------------------
- Graphite anode half-cell OCP from the Chen2020 parameter set
- NMC811 cathode half-cell OCP from the Chen2020 parameter set
- LFP cathode half-cell OCP from the Prada2013 parameter set

Notes
-----
- PyBaMM exposes built-in parameter sets through ``pybamm.parameter_sets`` and
  loads them via ``pybamm.ParameterValues(<name>)``.
- In parameter dictionaries, OCP functions are stored under keys such as
  ``"Negative electrode OCP [V]"`` and ``"Positive electrode OCP [V]"``.
- This file assumes the returned OCP functions accept array-like stoichiometry
  inputs on (0, 1) and return voltage values in volts.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from typing import Tuple, Dict, Callable

try:
    import pybamm
except ImportError as exc:
    raise ImportError(
        "PyBaMM is required for ocp_data_processing_pybamm.py. "
        "Install it with: pip install pybamm"
    ) from exc


# ============================================================================
# PyBaMM-backed OCP accessors
# ============================================================================

def _get_pybamm_ocp_function(electrode: str) -> Tuple[Callable, str, str, str]:
    """
    Return the PyBaMM OCP function and metadata for a requested electrode.

    Returns
    -------
    ocp_fn : callable
        OCP function from the PyBaMM parameter set.
    parameter_set : str
        Name of the PyBaMM parameter set.
    parameter_key : str
        Dictionary key used in the parameter set.
    electrode_role : str
        Either 'anode' or 'cathode'.
    """
    electrode = electrode.lower()

    if electrode == "graphite":
        parameter_set = "Chen2020"
        parameter_key = "Negative electrode OCP [V]"
        electrode_role = "anode"
    elif electrode == "nmc811":
        parameter_set = "Chen2020"
        parameter_key = "Positive electrode OCP [V]"
        electrode_role = "cathode"
    elif electrode == "lfp":
        parameter_set = "Prada2013"
        parameter_key = "Positive electrode OCP [V]"
        electrode_role = "cathode"
    else:
        raise ValueError(
            f"Unknown electrode '{electrode}'. Expected one of: graphite, nmc811, lfp"
        )

    params = pybamm.ParameterValues(parameter_set)
    ocp_fn = params[parameter_key]
    return ocp_fn, parameter_set, parameter_key, electrode_role



def pybamm_ocp(electrode: str, x: np.ndarray) -> np.ndarray:
    """
    Evaluate the PyBaMM OCP function for the selected electrode.

    Parameters
    ----------
    electrode : str
        'graphite', 'nmc811', or 'lfp'.
    x : np.ndarray
        Stoichiometry values in the open interval (0, 1).

    Returns
    -------
    np.ndarray
        OCP values in volts.
    """
    ocp_fn, _, _, _ = _get_pybamm_ocp_function(electrode)
    x = np.asarray(x, dtype=float)
    x_safe = np.clip(x, 1e-6, 1 - 1e-6)

    # PyBaMM bundled OCP functions are ordinary Python-callable functions in the
    # parameter dictionary. For numpy-array input they typically return ndarray-like
    # output; cast explicitly for downstream stability.
    U = ocp_fn(x_safe)
    return np.asarray(U, dtype=float)


# ============================================================================
# Dataset generation
# ============================================================================

def generate_ocp_dataset(
    electrode: str = "graphite",
    n_points: int = 200,
    noise_std: float = 0.003,
    seed: int = 42,
    x_range: Tuple[float, float] | None = None,
) -> pd.DataFrame:
    """
    Generate an OCP dataset using PyBaMM's built-in OCP functions.

    The returned dataframe keeps the same columns as the original code so the
    rest of the pipeline can remain unchanged.

    Parameters
    ----------
    electrode : str
        One of 'graphite', 'nmc811', or 'lfp'.
    n_points : int
        Number of data points.
    noise_std : float
        Standard deviation of additive Gaussian noise [V].
    seed : int
        Random seed.
    x_range : tuple, optional
        Stoichiometry range. If None, uses practical defaults for each electrode.

    Returns
    -------
    pd.DataFrame
        Columns:
        - x
        - U_measured
        - U_true
        - source
        - parameter_set
        - parameter_key
        - electrode_role
    """
    rng = np.random.RandomState(seed)
    electrode = electrode.lower()

    if electrode == "graphite":
        if x_range is None:
            x_range = (0.01, 0.99)
    elif electrode == "nmc811":
        if x_range is None:
            x_range = (0.25, 0.95)
    elif electrode == "lfp":
        if x_range is None:
            x_range = (0.01, 0.99)
    else:
        raise ValueError(
            f"Unknown electrode '{electrode}'. Expected one of: graphite, nmc811, lfp"
        )

    ocp_fn, parameter_set, parameter_key, electrode_role = _get_pybamm_ocp_function(electrode)

    x = np.linspace(x_range[0], x_range[1], n_points, dtype=float)
    U_true = np.asarray(ocp_fn(np.clip(x, 1e-6, 1 - 1e-6)), dtype=float)
    noise = rng.normal(loc=0.0, scale=noise_std, size=n_points)
    U_measured = U_true + noise

    df = pd.DataFrame(
        {
            "x": x,
            "U_measured": U_measured,
            "U_true": U_true,
            "source": "PyBaMM bundled parameter set",
            "parameter_set": parameter_set,
            "parameter_key": parameter_key,
            "electrode_role": electrode_role,
        }
    )
    return df


# ============================================================================
# Data preprocessing (kept same logic / API)
# ============================================================================

def preprocess_data(
    df: pd.DataFrame,
    train_frac: float = 0.7,
    val_frac: float = 0.15,
    normalize: bool = True,
    seed: int = 42,
) -> Dict:
    """
    Preprocess OCP data for model training.

    Keeps the original output structure used by the existing PINN / baseline code.
    """
    rng = np.random.RandomState(seed)

    x = df["x"].values.reshape(-1, 1)
    U = df["U_measured"].values.reshape(-1, 1)
    U_true = df["U_true"].values.reshape(-1, 1)

    n = len(df)
    idx = np.arange(n)
    rng.shuffle(idx)

    n_train = int(train_frac * n)
    n_val = int(val_frac * n)

    train_idx = idx[:n_train]
    val_idx = idx[n_train:n_train + n_val]
    test_idx = idx[n_train + n_val:]

    x_train, U_train = x[train_idx], U[train_idx]
    x_val, U_val = x[val_idx], U[val_idx]
    x_test, U_test = x[test_idx], U[test_idx]
    U_true_test = U_true[test_idx]

    out = {
        "x_train": x_train,
        "U_train": U_train,
        "x_val": x_val,
        "U_val": U_val,
        "x_test": x_test,
        "U_test": U_test,
        "U_true_test": U_true_test,
        "x_all": x,
        "U_all": U,
        "U_true_all": U_true,
        "df": df.copy(),
    }

    if normalize:
        x_min, x_max = x_train.min(), x_train.max()
        U_mean, U_std = U_train.mean(), U_train.std()
        if U_std < 1e-12:
            U_std = 1.0

        def x_norm(arr):
            return 2.0 * (arr - x_min) / (x_max - x_min) - 1.0

        def U_norm(arr):
            return (arr - U_mean) / U_std

        out.update(
            {
                "x_train_n": x_norm(x_train),
                "U_train_n": U_norm(U_train),
                "x_val_n": x_norm(x_val),
                "U_val_n": U_norm(U_val),
                "x_test_n": x_norm(x_test),
                "U_test_n": U_norm(U_test),
                "x_all_n": x_norm(x),
                "U_all_n": U_norm(U),
                "U_true_all_n": U_norm(U_true),
                "norm": {
                    "x_min": x_min,
                    "x_max": x_max,
                    "U_mean": U_mean,
                    "U_std": U_std,
                },
            }
        )
    else:
        out["norm"] = None

    return out


Overwriting ocp_data_processing_pybamm.py


In [15]:
%%writefile run_ocp_pinn_pybamm.py
"""
run_ocp_pinn_pybamm.py
======================
Main execution script for the OCP-PINN research workflow.

This version keeps the same pipeline logic as run_ocp_pinn.py, but replaces the
synthetic analytical dataset generator with a PyBaMM-backed generator that samples
OCP functions directly from built-in parameter sets.
"""

import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd

from ocp_data_processing_pybamm import generate_ocp_dataset, preprocess_data
from ocp_pinn_model import DirectOCPPINN, FreeEnergyPINN
from ocp_baselines import train_all_baselines
from ocp_evaluation import run_full_evaluation


def run_electrode(electrode: str, electrode_label: str, electrode_type: str,
                  output_dir: str, n_epochs: int = 5000):
    print(f"\n{'#'*70}")
    print(f"  Processing: {electrode_label}")
    print(f"{'#'*70}")

    print("\n[1/5] Generating OCP dataset from PyBaMM...")
    df = generate_ocp_dataset(electrode=electrode, n_points=200, noise_std=0.003)
    os.makedirs(f"{output_dir}/data", exist_ok=True)
    df.to_csv(f"{output_dir}/data/{electrode}_ocp.csv", index=False)
    print(f"  Data: {len(df)} points, U in [{df['U_true'].min():.3f}, {df['U_true'].max():.3f}] V")
    print(f"  Source: {df['parameter_set'].iloc[0]} / {df['parameter_key'].iloc[0]}")

    print("\n[2/5] Preprocessing data...")
    data = preprocess_data(df, train_frac=0.7, val_frac=0.15, normalize=True)
    print(f"  Train: {len(data['x_train'])}, Val: {len(data['x_val'])}, Test: {len(data['x_test'])}")

    print(f"\n[3/5] Training PINNs ({n_epochs} epochs)...")

    print("\n  --- Direct OCP PINN ---")
    pinn_direct = DirectOCPPINN(
        hidden_layers=[64, 64, 64, 64],
        lr=1e-3,
        w_data=1.0,
        w_physics=0.05,
        w_mono=0.005 if electrode_type == 'cathode' else 0.001,
        w_smooth=0.001,
        electrode_type=electrode_type,
        activation='tanh',
    )
    pinn_direct.train(data, n_epochs=n_epochs, n_colloc=300, verbose=True)
    print(f"  Direct PINN training time: {pinn_direct.training_time:.2f} s")

    print("\n  --- Free-Energy PINN ---")
    pinn_fe = FreeEnergyPINN(
        hidden_layers=[64, 64, 64, 64],
        lr=1e-3,
        w_data=1.0,
        w_convex=0.01,
        activation='tanh',
    )
    pinn_fe.train(data, n_epochs=n_epochs, n_colloc=300, verbose=True)
    print(f"  Free-Energy PINN training time: {pinn_fe.training_time:.2f} s")

    print(f"\n[4/5] Training baseline models...")
    baselines = train_all_baselines(data)

    print(f"\n[5/5] Evaluating all models...")
    table = run_full_evaluation(
        electrode_label,
        data,
        pinn_direct,
        pinn_fe,
        baselines,
        output_dir=f"{output_dir}/figures",
    )
    return table


def main():
    output_dir = "results_pybamm"
    os.makedirs(output_dir, exist_ok=True)

    n_epochs = 5000
    electrodes = [
        ("graphite", "Graphite (Li_xC_6)", "anode"),
        ("nmc811", "NMC811 (Li_xNi_{0.8}Mn_{0.1}Co_{0.1}O_2)", "cathode"),
        ("lfp", "LFP (Li_xFePO_4)", "cathode"),
    ]

    all_tables = []
    for electrode, label, etype in electrodes:
        table = run_electrode(electrode, label, etype, output_dir, n_epochs)
        table["Electrode"] = label
        all_tables.append(table)

    combined = pd.concat(all_tables, ignore_index=True)
    combined.to_csv(f"{output_dir}/combined_comparison.csv", index=False)

    print(f"\n\n{'='*70}")
    print("  COMBINED RESULTS — ALL ELECTRODES")
    print(f"{'='*70}")
    print(combined.to_string(index=False))
    print(f"\n\nAll results saved to {output_dir}/")


if __name__ == "__main__":
    main()


Overwriting run_ocp_pinn_pybamm.py


In [16]:
# Original modules retained unchanged
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
%%writefile ocp_pinn_model.py
"""
ocp_pinn_model.py
=================
Physics-Informed Neural Network (PINN) for Open-Circuit Potential (OCP) modeling.

The PINN learns the OCP U(x) as a function of stoichiometry x while enforcing
thermodynamic constraints derived from the Gibbs free energy of mixing.

Two PINN formulations are implemented:

1. **Direct OCP PINN**: Learns U(x) directly with physics-informed regularization
   that penalizes violations of thermodynamic consistency:
   - Monotonicity: dU/dx <= 0 for cathodes (or appropriate sign for anodes)
   - Smoothness: penalize large second derivatives (physical plausibility)
   - Boundary behavior: physically meaningful boundary conditions

2. **Free-Energy PINN**: Learns the Gibbs free energy G(x) and derives U(x) 
   through automatic differentiation: U(x) = -(1/F) * dG/dx.
   This inherently enforces thermodynamic consistency because the OCP is 
   derived from a single scalar potential.

The governing equation relates OCP to the chemical potential:
    U(x) = U0 - (RT/F) * ln(x/(1-x)) + U_excess(x)

where U_excess(x) = (1/F) * d(g_excess)/dx and g_excess is the excess Gibbs
energy of mixing. The PINN residual penalizes departure from this relation.

References:
    Raissi et al., J. Comput. Phys. 378 (2019) 686.
    Yao & Viswanathan, J. Phys. Chem. Lett. 15 (2024) 1105.
    Ferguson & Bazant, J. Electrochem. Soc. 159 (2012) A1967.
"""

import torch
import torch.nn as nn
import numpy as np
import time
from typing import Dict, Optional, Tuple


# ============================================================================
# Constants
# ============================================================================
R_GAS = 8.314462     # J/(mol·K)
F_CONST = 96485.33   # C/mol
T_REF = 298.15       # K (25 °C)


# ============================================================================
# Neural network architecture
# ============================================================================

class PINN_Net(nn.Module):
    """
    Fully connected neural network with smooth activation functions.
    
    Uses tanh activation throughout to ensure smooth outputs and well-defined
    derivatives, which is critical for physics-informed loss terms that 
    involve first and second derivatives of the network output.
    
    Parameters
    ----------
    input_dim : int
        Dimension of input (1 for U(x), 2 for U(x,T)).
    output_dim : int
        Dimension of output (1 for scalar OCP or free energy).
    hidden_layers : list of int
        Number of neurons per hidden layer.
    activation : str
        Activation function: 'tanh', 'silu', or 'gelu'.
    """
    
    def __init__(self, input_dim=1, output_dim=1, 
                 hidden_layers=[64, 64, 64, 64], activation='tanh'):
        super().__init__()
        
        layers = []
        in_features = input_dim
        
        # Select activation function
        act_fn = {
            'tanh': nn.Tanh,
            'silu': nn.SiLU,
            'gelu': nn.GELU,
        }[activation]
        
        for h in hidden_layers:
            layers.append(nn.Linear(in_features, h))
            layers.append(act_fn())
            in_features = h
        
        layers.append(nn.Linear(in_features, output_dim))
        self.net = nn.Sequential(*layers)
        
        # Xavier initialization for better training stability
        self._init_weights()
    
    def _init_weights(self):
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, x):
        return self.net(x)


# ============================================================================
# Direct OCP PINN
# ============================================================================

class DirectOCPPINN:
    """
    PINN that learns OCP U(x) directly with thermodynamic constraints.
    
    Loss function:
        L = w_data * L_data + w_physics * L_physics + w_mono * L_mono + w_smooth * L_smooth
    
    where:
    - L_data: MSE between predicted and observed OCP values
    - L_physics: residual of the governing thermodynamic relation
    - L_mono: penalty for violating monotonicity (dU/dx should have correct sign)
    - L_smooth: penalty for excessive curvature (d²U/dx²)
    
    Parameters
    ----------
    hidden_layers : list
        Network architecture.
    lr : float
        Learning rate.
    w_data : float
        Weight for data fitting loss.
    w_physics : float
        Weight for physics residual loss.
    w_mono : float
        Weight for monotonicity constraint.
    w_smooth : float
        Weight for smoothness regularization.
    electrode_type : str
        'cathode' (dU/dx < 0) or 'anode' (non-monotonic, different constraints).
    """
    
    def __init__(self, hidden_layers=[64, 64, 64, 64], lr=1e-3,
                 w_data=1.0, w_physics=0.1, w_mono=0.01, w_smooth=0.001,
                 electrode_type='cathode', activation='tanh'):
        
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.net = PINN_Net(1, 1, hidden_layers, activation).to(self.device)
        self.optimizer = torch.optim.Adam(self.net.parameters(), lr=lr)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, patience=500, factor=0.5, min_lr=1e-6)
        
        self.w_data = w_data
        self.w_physics = w_physics
        self.w_mono = w_mono
        self.w_smooth = w_smooth
        self.electrode_type = electrode_type
        
        self.train_losses = []
        self.val_losses = []
        self.training_time = 0.0
    
    def _compute_derivatives(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Compute U, dU/dx, and d²U/dx² using automatic differentiation."""
        x.requires_grad_(True)
        U = self.net(x)
        
        dU_dx = torch.autograd.grad(
            U, x, grad_outputs=torch.ones_like(U),
            create_graph=True, retain_graph=True
        )[0]
        
        d2U_dx2 = torch.autograd.grad(
            dU_dx, x, grad_outputs=torch.ones_like(dU_dx),
            create_graph=True, retain_graph=True
        )[0]
        
        return U, dU_dx, d2U_dx2
    
    def _physics_residual(self, x_colloc: torch.Tensor) -> torch.Tensor:
        """
        Compute the physics residual.
        
        The thermodynamic relation for the ideal part of OCP is:
            U_ideal(x) = U0 - (RT/F) * ln(x / (1-x))
        
        The derivative of this ideal contribution is:
            dU_ideal/dx = -(RT/F) * [1/x + 1/(1-x)]
                        = -(RT/F) / [x(1-x)]
        
        The physics residual checks if the learned dU/dx is consistent with
        having a contribution from the ideal entropy of mixing.
        Specifically, we penalize: |d²U/dx² + (RT/F) * (1-2x)/[x(1-x)]²|
        which would be zero for a pure ideal solution model.
        
        For real materials, the excess contribution makes this non-zero, but
        we use a soft penalty to encourage thermodynamic plausibility.
        """
        x_colloc = x_colloc.requires_grad_(True)
        U, dU_dx, d2U_dx2 = self._compute_derivatives(x_colloc)
        
        # Ideal solution second derivative: d²U_ideal/dx² = (RT/F)(1-2x)/[x(1-x)]²
        x_safe = torch.clamp(x_colloc, 1e-4, 1 - 1e-4)
        d2U_ideal = (R_GAS * T_REF / F_CONST) * (1 - 2*x_safe) / (x_safe * (1 - x_safe))**2
        
        # The excess part d²U_excess/dx² should be smooth (bounded polynomial)
        # Residual: the non-ideal part of the curvature should be small/smooth
        residual = d2U_dx2 - d2U_ideal
        
        return residual
    
    def _monotonicity_loss(self, x_colloc: torch.Tensor) -> torch.Tensor:
        """Penalize violations of monotonicity."""
        _, dU_dx, _ = self._compute_derivatives(x_colloc)
        
        if self.electrode_type == 'cathode':
            # For cathodes: dU/dx <= 0 (OCP decreases with lithiation)
            violation = torch.relu(dU_dx)
        else:
            # For anodes like graphite, monotonicity is more complex
            # due to staging plateaus. Use a weaker constraint.
            violation = torch.relu(dU_dx - 0.5)  # Allow some positive slopes
        
        return torch.mean(violation**2)
    
    def _smoothness_loss(self, x_colloc: torch.Tensor) -> torch.Tensor:
        """Penalize excessive curvature for smoothness."""
        _, _, d2U_dx2 = self._compute_derivatives(x_colloc)
        return torch.mean(d2U_dx2**2)
    
    def train(self, data: Dict, n_epochs: int = 5000, n_colloc: int = 500,
              verbose: bool = True) -> Dict:
        """
        Train the PINN.
        
        Parameters
        ----------
        data : dict
            Preprocessed data from ocp_data_processing.preprocess_data().
        n_epochs : int
            Number of training epochs.
        n_colloc : int
            Number of collocation points for physics loss.
        verbose : bool
            Print training progress.
        
        Returns
        -------
        history : dict
            Training and validation loss history.
        """
        # Convert data to tensors
        x_train = torch.FloatTensor(data['x_train']).to(self.device)
        U_train = torch.FloatTensor(data['U_train']).to(self.device)
        x_val = torch.FloatTensor(data['x_val']).to(self.device)
        U_val = torch.FloatTensor(data['U_val']).to(self.device)
        
        start_time = time.time()
        
        for epoch in range(n_epochs):
            self.net.train()
            self.optimizer.zero_grad()
            
            # Data loss
            U_pred = self.net(x_train)
            loss_data = torch.mean((U_pred - U_train)**2)
            
            # Collocation points for physics (uniformly sampled in [0, 1])
            x_colloc = torch.rand(n_colloc, 1, device=self.device)
            x_colloc = x_colloc * 0.96 + 0.02  # Avoid boundaries
            
            # Physics residual loss
            residual = self._physics_residual(x_colloc)
            loss_physics = torch.mean(residual**2)
            
            # Monotonicity loss
            loss_mono = self._monotonicity_loss(x_colloc)
            
            # Smoothness loss
            loss_smooth = self._smoothness_loss(x_colloc)
            
            # Total loss
            loss = (self.w_data * loss_data 
                    + self.w_physics * loss_physics
                    + self.w_mono * loss_mono
                    + self.w_smooth * loss_smooth)
            
            loss.backward()
            self.optimizer.step()
            
            # Validation
            self.net.eval()
            with torch.no_grad():
                U_val_pred = self.net(x_val)
                val_loss = torch.mean((U_val_pred - U_val)**2).item()
            
            self.train_losses.append(loss.item())
            self.val_losses.append(val_loss)
            self.scheduler.step(val_loss)
            
            if verbose and (epoch + 1) % 1000 == 0:
                print(f"Epoch {epoch+1}/{n_epochs} | "
                      f"Loss: {loss.item():.6f} | "
                      f"Data: {loss_data.item():.6f} | "
                      f"Physics: {loss_physics.item():.6f} | "
                      f"Val: {val_loss:.6f}")
        
        self.training_time = time.time() - start_time
        
        return {
            'train_losses': self.train_losses,
            'val_losses': self.val_losses,
            'training_time': self.training_time
        }
    
    def predict(self, x: np.ndarray) -> np.ndarray:
        """Predict OCP for given stoichiometry values."""
        self.net.eval()
        with torch.no_grad():
            x_tensor = torch.FloatTensor(x.reshape(-1, 1)).to(self.device)
            U_pred = self.net(x_tensor).cpu().numpy().flatten()
        return U_pred
    
    def predict_with_derivatives(self, x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Predict OCP and its derivative dU/dx."""
        self.net.eval()
        x_tensor = torch.FloatTensor(x.reshape(-1, 1)).to(self.device)
        x_tensor.requires_grad_(True)
        
        U = self.net(x_tensor)
        dU_dx = torch.autograd.grad(
            U, x_tensor, grad_outputs=torch.ones_like(U),
            create_graph=False
        )[0]
        
        return (U.detach().cpu().numpy().flatten(),
                dU_dx.detach().cpu().numpy().flatten())


# ============================================================================
# Free-Energy PINN
# ============================================================================

class FreeEnergyPINN:
    """
    PINN that learns the Gibbs free energy G(x) and derives OCP via differentiation.
    
    The OCP is computed as:
        U(x) = -(1/F) * dG/dx
    
    This formulation inherently enforces thermodynamic consistency because:
    1. OCP is derived from a single scalar potential (the free energy).
    2. The convexity of G(x) directly maps to monotonicity of U(x).
    3. Phase coexistence (plateaus) arise naturally from common tangent construction.
    
    The network learns G(x) directly, and U(x) is obtained through PyTorch's
    automatic differentiation. Physics constraints on G(x) include:
    - G should contain an ideal mixing entropy contribution
    - G should be smooth and differentiable
    - Boundary conditions: G(0) and G(1) correspond to pure phases
    
    Parameters
    ----------
    hidden_layers : list
        Network architecture.
    lr : float
        Learning rate.
    w_data : float
        Weight for OCP data fitting loss.
    w_convex : float
        Weight for convexity regularization on G(x).
    w_boundary : float
        Weight for boundary condition loss.
    """
    
    def __init__(self, hidden_layers=[64, 64, 64, 64], lr=1e-3,
                 w_data=1.0, w_convex=0.01, w_boundary=0.1,
                 activation='tanh'):
        
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.net = PINN_Net(1, 1, hidden_layers, activation).to(self.device)
        self.optimizer = torch.optim.Adam(self.net.parameters(), lr=lr)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, patience=500, factor=0.5, min_lr=1e-6)
        
        self.w_data = w_data
        self.w_convex = w_convex
        self.w_boundary = w_boundary
        
        self.train_losses = []
        self.val_losses = []
        self.training_time = 0.0
    
    def _compute_ocp_from_G(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Compute OCP from free energy via differentiation.
        
        U(x) = -(1/F) * dG/dx
        
        Also compute d²G/dx² for convexity check.
        """
        x.requires_grad_(True)
        G = self.net(x)
        
        dG_dx = torch.autograd.grad(
            G, x, grad_outputs=torch.ones_like(G),
            create_graph=True, retain_graph=True
        )[0]
        
        d2G_dx2 = torch.autograd.grad(
            dG_dx, x, grad_outputs=torch.ones_like(dG_dx),
            create_graph=True, retain_graph=True
        )[0]
        
        # OCP from thermodynamics: U = -(1/F) * dG/dx
        # Since we work with normalized values, the scaling is absorbed 
        # into the network weights
        U = -dG_dx  # Simplified: F scaling absorbed into training
        
        return U, G, d2G_dx2
    
    def train(self, data: Dict, n_epochs: int = 5000, n_colloc: int = 500,
              verbose: bool = True) -> Dict:
        """Train the Free-Energy PINN."""
        x_train = torch.FloatTensor(data['x_train']).to(self.device)
        U_train = torch.FloatTensor(data['U_train']).to(self.device)
        x_val = torch.FloatTensor(data['x_val']).to(self.device)
        U_val = torch.FloatTensor(data['U_val']).to(self.device)
        
        start_time = time.time()
        
        for epoch in range(n_epochs):
            self.net.train()
            self.optimizer.zero_grad()
            
            # Data loss: compare derived OCP with observations
            U_pred, _, _ = self._compute_ocp_from_G(x_train)
            loss_data = torch.mean((U_pred - U_train)**2)
            
            # Collocation points for physics constraints
            x_colloc = torch.rand(n_colloc, 1, device=self.device)
            x_colloc = x_colloc * 0.96 + 0.02
            
            # Convexity: d²G/dx² >= 0 for thermodynamic stability
            _, _, d2G_dx2 = self._compute_ocp_from_G(x_colloc)
            loss_convex = torch.mean(torch.relu(-d2G_dx2)**2)
            
            # Total loss
            loss = (self.w_data * loss_data 
                    + self.w_convex * loss_convex)
            
            loss.backward()
            self.optimizer.step()
            
            # Validation
            self.net.eval()
            with torch.no_grad():
                # For validation, we need gradients, so use eval but enable grad temporarily
                pass
            
            # Re-enable grad for validation OCP computation
            U_val_pred, _, _ = self._compute_ocp_from_G(x_val)
            val_loss = torch.mean((U_val_pred.detach() - U_val)**2).item()
            
            self.train_losses.append(loss.item())
            self.val_losses.append(val_loss)
            self.scheduler.step(val_loss)
            
            if verbose and (epoch + 1) % 1000 == 0:
                print(f"Epoch {epoch+1}/{n_epochs} | "
                      f"Loss: {loss.item():.6f} | "
                      f"Data: {loss_data.item():.6f} | "
                      f"Convex: {loss_convex.item():.6f} | "
                      f"Val: {val_loss:.6f}")
        
        self.training_time = time.time() - start_time
        
        return {
            'train_losses': self.train_losses,
            'val_losses': self.val_losses,
            'training_time': self.training_time
        }
    
    def predict(self, x: np.ndarray) -> np.ndarray:
        """Predict OCP for given stoichiometry values."""
        self.net.eval()
        x_tensor = torch.FloatTensor(x.reshape(-1, 1)).to(self.device)
        x_tensor.requires_grad_(True)
        
        G = self.net(x_tensor)
        dG_dx = torch.autograd.grad(
            G, x_tensor, grad_outputs=torch.ones_like(G),
            create_graph=False
        )[0]
        
        U = -dG_dx
        return U.detach().cpu().numpy().flatten()
    
    def predict_free_energy(self, x: np.ndarray) -> np.ndarray:
        """Predict the learned free energy G(x)."""
        self.net.eval()
        with torch.no_grad():
            x_tensor = torch.FloatTensor(x.reshape(-1, 1)).to(self.device)
            G = self.net(x_tensor).cpu().numpy().flatten()
        return G
    
    def predict_with_derivatives(self, x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Predict OCP and dU/dx."""
        self.net.eval()
        x_tensor = torch.FloatTensor(x.reshape(-1, 1)).to(self.device)
        x_tensor.requires_grad_(True)
        
        G = self.net(x_tensor)
        dG_dx = torch.autograd.grad(
            G, x_tensor, grad_outputs=torch.ones_like(G),
            create_graph=True, retain_graph=True
        )[0]
        
        d2G_dx2 = torch.autograd.grad(
            dG_dx, x_tensor, grad_outputs=torch.ones_like(dG_dx),
            create_graph=False
        )[0]
        
        U = -dG_dx
        dU_dx = -d2G_dx2
        
        return (U.detach().cpu().numpy().flatten(),
                dU_dx.detach().cpu().numpy().flatten())


# ============================================================================
# Convenience function
# ============================================================================

def build_pinn(formulation: str = 'direct', electrode_type: str = 'cathode',
               **kwargs) -> object:
    """
    Factory function to create a PINN model.
    
    Parameters
    ----------
    formulation : str
        'direct' for DirectOCPPINN, 'free_energy' for FreeEnergyPINN.
    electrode_type : str
        'cathode' or 'anode'.
    **kwargs
        Additional arguments passed to the PINN constructor.
    
    Returns
    -------
    model : DirectOCPPINN or FreeEnergyPINN
    """
    if formulation == 'direct':
        return DirectOCPPINN(electrode_type=electrode_type, **kwargs)
    elif formulation == 'free_energy':
        return FreeEnergyPINN(**kwargs)
    else:
        raise ValueError(f"Unknown formulation: {formulation}")


if __name__ == "__main__":
    print("PINN models loaded successfully.")
    print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
    
    # Quick test
    net = PINN_Net(1, 1, [32, 32])
    x_test = torch.randn(10, 1)
    y_test = net(x_test)
    print(f"Test output shape: {y_test.shape}")


Overwriting ocp_pinn_model.py


In [18]:
%%writefile ocp_baselines.py
"""
ocp_baselines.py
================
Baseline models for OCP fitting — conventional methods compared against the PINN.

Implements the following baselines:
1. Polynomial regression (degree 8 and 12)
2. Cubic spline interpolation
3. Nernst-type analytical fit (ideal solution + polynomial correction)
4. Redlich-Kister thermodynamic fit
5. Gaussian Process Regression (GPR)

Each baseline uses the same train/test split as the PINN for fair comparison.

References:
    Plett, Battery Management Systems, Vol. 1 (Artech House, 2015).
    Yao & Viswanathan, J. Phys. Chem. Lett. 15 (2024) 1105.
    Karthikeyan et al., J. Power Sources 185 (2008) 1398.
"""

import numpy as np
import time
from scipy.interpolate import CubicSpline
from scipy.optimize import curve_fit
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, Matern
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from typing import Dict, Tuple, Optional


# ============================================================================
# Constants
# ============================================================================
R_GAS = 8.314462
F_CONST = 96485.33
T_REF = 298.15


# ============================================================================
# 1. Polynomial regression
# ============================================================================

class PolynomialOCPModel:
    """
    OCP model using polynomial regression.
    
    Polynomial fitting is the simplest and most widely used empirical approach 
    for OCP modeling. High-degree polynomials (n >= 8) are typically needed 
    to capture the non-linear features of intercalation electrodes.
    
    Limitation: Does not enforce thermodynamic consistency or monotonicity.
    Can produce oscillatory artifacts at boundaries (Runge's phenomenon).
    
    References:
        Doyle et al., J. Electrochem. Soc. 143 (1996) 1890.
        Sikha et al., J. Electrochem. Soc. 152 (2005) A1682.
    
    Parameters
    ----------
    degree : int
        Polynomial degree (typically 8-16 for battery OCP).
    """
    
    def __init__(self, degree: int = 12):
        self.degree = degree
        self.model = make_pipeline(
            PolynomialFeatures(degree), 
            LinearRegression()
        )
        self.training_time = 0.0
        self.name = f"Polynomial (deg {degree})"
    
    def train(self, x_train: np.ndarray, U_train: np.ndarray):
        """Fit the polynomial model."""
        start = time.time()
        self.model.fit(x_train.reshape(-1, 1), U_train.ravel())
        self.training_time = time.time() - start
    
    def predict(self, x: np.ndarray) -> np.ndarray:
        """Predict OCP."""
        return self.model.predict(x.reshape(-1, 1))


# ============================================================================
# 2. Cubic spline interpolation
# ============================================================================

class SplineOCPModel:
    """
    OCP model using cubic spline interpolation.
    
    Splines provide smooth, piecewise-polynomial fits that pass through data
    points. They are highly flexible and widely used in battery management 
    systems for OCV-SOC lookup tables.
    
    Limitation: Cannot extrapolate beyond the training data range. Does not 
    enforce monotonicity, and the resulting function gradients (dU/dx) may 
    not be thermodynamically meaningful.
    
    Reference:
        Yao & Viswanathan, J. Phys. Chem. Lett. 15 (2024) 1105 (discussion of 
        spline-based models and their thermodynamic limitations).
    """
    
    def __init__(self):
        self.spline = None
        self.training_time = 0.0
        self.name = "Cubic Spline"
    
    def train(self, x_train: np.ndarray, U_train: np.ndarray):
        """Fit the cubic spline."""
        start = time.time()
        # Sort data by x for spline construction
        sort_idx = np.argsort(x_train.ravel())
        x_sorted = x_train.ravel()[sort_idx]
        U_sorted = U_train.ravel()[sort_idx]
        
        # Remove duplicate x values (keep mean U)
        unique_x, unique_idx = np.unique(x_sorted, return_index=True)
        unique_U = np.array([U_sorted[x_sorted == xi].mean() for xi in unique_x])
        
        self.spline = CubicSpline(unique_x, unique_U)
        self.x_min = unique_x.min()
        self.x_max = unique_x.max()
        self.training_time = time.time() - start
    
    def predict(self, x: np.ndarray) -> np.ndarray:
        """Predict OCP (clipped to training range for extrapolation safety)."""
        x_clipped = np.clip(x.ravel(), self.x_min, self.x_max)
        return self.spline(x_clipped)


# ============================================================================
# 3. Nernst-type analytical fit
# ============================================================================

class NernstOCPModel:
    """
    OCP model combining the Nernst equation with polynomial correction.
    
    The model expresses OCP as:
        U(x) = U0 - (RT/F) * ln(x/(1-x)) + sum_i a_i * x^i
    
    The first two terms represent the ideal thermodynamic contribution from
    entropy of mixing (Nernst equation for a single-site lattice model).
    The polynomial correction captures deviations due to non-ideal 
    interactions (excess Gibbs energy).
    
    This is physically more meaningful than pure polynomial fitting because
    it correctly captures the logarithmic divergence at x -> 0 and x -> 1.
    
    Parameters
    ----------
    poly_degree : int
        Degree of the correction polynomial.
    T : float
        Temperature in Kelvin.
    """
    
    def __init__(self, poly_degree: int = 6, T: float = T_REF):
        self.poly_degree = poly_degree
        self.T = T
        self.params = None
        self.training_time = 0.0
        self.name = f"Nernst + Poly (deg {poly_degree})"
    
    def _nernst_ideal(self, x: np.ndarray) -> np.ndarray:
        """Ideal contribution: -(RT/F) * ln(x/(1-x))."""
        x_safe = np.clip(x, 1e-6, 1 - 1e-6)
        return -(R_GAS * self.T / F_CONST) * np.log(x_safe / (1 - x_safe))
    
    def _model_func(self, x, *params):
        """Full model: U0 + Nernst + polynomial."""
        U0 = params[0]
        poly_coeffs = params[1:]
        
        U = U0 + self._nernst_ideal(x)
        for i, a in enumerate(poly_coeffs):
            U = U + a * x**(i+1)
        return U
    
    def train(self, x_train: np.ndarray, U_train: np.ndarray):
        """Fit the Nernst + polynomial model using nonlinear least squares."""
        start = time.time()
        
        x = x_train.ravel()
        U = U_train.ravel()
        
        # Initial guess: U0 from data midpoint, zero polynomial coefficients
        p0 = [np.mean(U)] + [0.0] * self.poly_degree
        
        try:
            self.params, _ = curve_fit(
                self._model_func, x, U, p0=p0,
                maxfev=10000
            )
        except RuntimeError:
            # If optimization fails, fall back to simpler fit
            self.params = p0
        
        self.training_time = time.time() - start
    
    def predict(self, x: np.ndarray) -> np.ndarray:
        """Predict OCP."""
        return self._model_func(x.ravel(), *self.params)


# ============================================================================
# 4. Redlich-Kister thermodynamic fit
# ============================================================================

class RedlichKisterOCPModel:
    """
    OCP model based on Redlich-Kister expansion of excess Gibbs energy.
    
    The thermodynamic model expresses OCP as:
        U(x) = U0 - (RT/F) * ln(x/(1-x)) + (1/F) * d(g_excess)/dx
    
    where g_excess = x*(1-x) * sum_{i=0}^{N} Omega_i * (1-2x)^i
    
    This is the standard thermodynamic approach for modeling OCP of 
    intercalation materials. The Omega_i coefficients have units of J/mol 
    and represent the strength of lithium-vacancy interactions.
    
    Limitation: Does not enforce monotonicity. For materials with phase 
    separation (e.g., LFP), the regular R-K model produces non-physical 
    oscillations unless combined with common tangent construction.
    
    References:
        Karthikeyan et al., J. Power Sources 185 (2008) 1398.
        Plett, Battery Management Systems, Vol. 1 (Artech House, 2015).
        Yao & Viswanathan, J. Phys. Chem. Lett. 15 (2024) 1105.
    
    Parameters
    ----------
    n_terms : int
        Number of R-K coefficients (typically 3-8).
    T : float
        Temperature in Kelvin.
    """
    
    def __init__(self, n_terms: int = 6, T: float = T_REF):
        self.n_terms = n_terms
        self.T = T
        self.params = None
        self.training_time = 0.0
        self.name = f"Redlich-Kister ({n_terms} terms)"
    
    def _rk_model(self, x, U0, *omega):
        """Compute OCP from R-K expansion."""
        x_safe = np.clip(x, 1e-6, 1 - 1e-6)
        
        # Ideal mixing term
        U = U0 - (R_GAS * self.T / F_CONST) * np.log(x_safe / (1 - x_safe))
        
        # Excess chemical potential from R-K expansion
        # mu_excess = d(g_excess * N_total) / dN_Li
        # For binary on single lattice:
        # mu_excess/F = (1/F) * sum_i omega_i * [(1-2x)^(i+1) - 2*i*x*(1-x)*(1-2x)^(i-1)]
        z = 1 - 2 * x_safe
        for i, om in enumerate(omega):
            if i == 0:
                U += (om / F_CONST) * z
            else:
                U += (om / F_CONST) * (z**(i+1) - 2*i*x_safe*(1-x_safe)*z**(i-1))
        
        return U
    
    def train(self, x_train: np.ndarray, U_train: np.ndarray):
        """Fit the R-K model."""
        start = time.time()
        
        x = x_train.ravel()
        U = U_train.ravel()
        
        # Initial guess
        p0 = [np.mean(U)] + [0.0] * self.n_terms
        
        try:
            self.params, _ = curve_fit(
                self._rk_model, x, U, p0=p0,
                maxfev=20000
            )
        except RuntimeError:
            self.params = p0
        
        self.training_time = time.time() - start
    
    def predict(self, x: np.ndarray) -> np.ndarray:
        """Predict OCP."""
        return self._rk_model(x.ravel(), *self.params)


# ============================================================================
# 5. Gaussian Process Regression
# ============================================================================

class GPR_OCPModel:
    """
    OCP model using Gaussian Process Regression.
    
    GPR is a non-parametric Bayesian regression method that provides both
    a mean prediction and uncertainty estimates. It is well-suited for 
    interpolation of smooth functions like OCP curves.
    
    The Matern kernel is chosen for its ability to model varying degrees 
    of smoothness, and the WhiteKernel accounts for observation noise.
    
    Limitation: Computational cost scales as O(n³), making it impractical 
    for very large datasets. No built-in physics constraints.
    """
    
    def __init__(self):
        kernel = Matern(length_scale=0.1, nu=2.5) + WhiteKernel(noise_level=1e-4)
        self.model = GaussianProcessRegressor(
            kernel=kernel,
            n_restarts_optimizer=5,
            normalize_y=True
        )
        self.training_time = 0.0
        self.name = "Gaussian Process (Matern)"
    
    def train(self, x_train: np.ndarray, U_train: np.ndarray):
        """Fit the GPR model."""
        start = time.time()
        self.model.fit(x_train.reshape(-1, 1), U_train.ravel())
        self.training_time = time.time() - start
    
    def predict(self, x: np.ndarray) -> np.ndarray:
        """Predict OCP (mean prediction)."""
        return self.model.predict(x.reshape(-1, 1))
    
    def predict_with_uncertainty(self, x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Predict OCP with uncertainty bounds."""
        mean, std = self.model.predict(x.reshape(-1, 1), return_std=True)
        return mean, std


# ============================================================================
# Factory and utilities
# ============================================================================

def get_all_baselines() -> list:
    """Return a list of all baseline models."""
    return [
        PolynomialOCPModel(degree=8),
        PolynomialOCPModel(degree=12),
        SplineOCPModel(),
        NernstOCPModel(poly_degree=6),
        RedlichKisterOCPModel(n_terms=6),
        GPR_OCPModel(),
    ]


def train_all_baselines(data: Dict) -> list:
    """
    Train all baseline models on the same data.
    
    Parameters
    ----------
    data : dict
        Preprocessed data dictionary.
    
    Returns
    -------
    models : list of trained baseline models
    """
    models = get_all_baselines()
    
    for model in models:
        print(f"Training {model.name}...")
        model.train(data['x_train'], data['U_train'])
        print(f"  Training time: {model.training_time:.4f} s")
    
    return models


if __name__ == "__main__":
    # Quick test with synthetic data
    x = np.linspace(0.01, 0.99, 100)
    U = 3.4 - 0.5 * x + 0.1 * np.sin(5 * x)
    
    for ModelClass in [PolynomialOCPModel, SplineOCPModel, NernstOCPModel,
                       RedlichKisterOCPModel, GPR_OCPModel]:
        if ModelClass == PolynomialOCPModel:
            model = ModelClass(degree=8)
        elif ModelClass == RedlichKisterOCPModel:
            model = ModelClass(n_terms=4)
        elif ModelClass == NernstOCPModel:
            model = ModelClass(poly_degree=4)
        else:
            model = ModelClass()
        
        model.train(x, U)
        U_pred = model.predict(x)
        rmse = np.sqrt(np.mean((U_pred - U)**2))
        print(f"{model.name}: RMSE = {rmse:.6f}, Time = {model.training_time:.4f}s")


Overwriting ocp_baselines.py


In [25]:
%%writefile ocp_evaluation.py

"""
ocp_evaluation.py
=================
Evaluation metrics, comparison tables, and publication-quality figures for 
OCP model benchmarking.

Computes:
- RMSE, MAE, R², max absolute error
- Training and inference time
- Derivative consistency (dU/dx)
- Boundary behavior and extrapolation assessment

Generates:
- OCP prediction vs data plots
- Residual/error plots
- Derivative comparison plots
- Free energy profiles (for Free-Energy PINN)
- Comprehensive comparison table
- Training loss curves
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 11
matplotlib.rcParams['axes.labelsize'] = 12
matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['savefig.dpi'] = 200
matplotlib.rcParams['savefig.bbox'] = 'tight'

import time
from typing import Dict, List, Optional
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# Denormalization helpers (defined inline; ocp_data_processing_pybamm uses
# z-score for U and min-max for x)
# ---------------------------------------------------------------------------
def denormalize_U(U_norm, U_mean_or_min, U_std_or_max):
    """Inverse of z-score normalisation: U = U_norm * std + mean."""
    return U_norm * U_std_or_max + U_mean_or_min

def denormalize_x(x_norm, x_min, x_max):
    """Inverse of [-1,1] min-max normalisation: x = (x_norm+1)/2*(max-min)+min."""
    return (x_norm + 1.0) / 2.0 * (x_max - x_min) + x_min

def unique_color_cycle(n):
    palettes = []
    for cmap_name in ["tab20", "tab20b", "tab20c", "Set3", "Paired", "Dark2"]:
        cmap = plt.get_cmap(cmap_name)
        if hasattr(cmap, "colors"):
            palettes.extend(list(cmap.colors))

    unique = []
    seen = set()
    for c in palettes:
        key = tuple(float(x) for x in c)
        if key not in seen:
            seen.add(key)
            unique.append(c)

    if n > len(unique):
        raise ValueError(f"Requested {n} unique colors, but only {len(unique)} are available.")
    return unique[:n]


# ============================================================================
# Metrics
# ============================================================================

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict:
    """
    Compute regression metrics for OCP prediction.
    
    Parameters
    ----------
    y_true, y_pred : np.ndarray
        True and predicted OCP values (same units, same scale).
    
    Returns
    -------
    metrics : dict
        RMSE, MAE, R2, max_error.
    """
    residuals = y_true - y_pred
    
    rmse = np.sqrt(np.mean(residuals**2))
    mae = np.mean(np.abs(residuals))
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    r2 = 1 - ss_res / (ss_tot + 1e-10)
    max_err = np.max(np.abs(residuals))
    
    return {
        'RMSE (V)': rmse,
        'MAE (V)': mae,
        'R²': r2,
        'Max Error (V)': max_err,
    }


def compute_inference_time(model, x: np.ndarray, n_runs: int = 100) -> float:
    """
    Measure average inference time.
    
    Parameters
    ----------
    model : object with predict() method
    x : np.ndarray
        Input stoichiometry values.
    n_runs : int
        Number of repetitions for timing.
    
    Returns
    -------
    avg_time_ms : float
        Average inference time in milliseconds.
    """
    # Warm-up
    _ = model.predict(x)
    
    start = time.time()
    for _ in range(n_runs):
        _ = model.predict(x)
    total = time.time() - start
    
    return (total / n_runs) * 1000  # ms


# ============================================================================
# Comparison table
# ============================================================================

def build_comparison_table(
    models: list,
    model_names: list,
    x_test: np.ndarray,
    U_test: np.ndarray,
    training_times: list,
    data_for_normalize: Optional[Dict] = None,
) -> pd.DataFrame:
    """
    Build a comprehensive comparison table of all models.
    
    Parameters
    ----------
    models : list
        Trained models with predict() methods.
    model_names : list of str
        Names for each model.
    x_test : np.ndarray
        Test stoichiometry (normalized or raw, matching model expectations).
    U_test : np.ndarray
        Test OCP values (same normalization as model output).
    training_times : list of float
        Training time for each model in seconds.
    data_for_normalize : dict, optional
        If provided, denormalize predictions before computing metrics.
    
    Returns
    -------
    table : pd.DataFrame
    """
    rows = []
    
    for model, name, t_time in zip(models, model_names, training_times):
        U_pred = model.predict(x_test)
        
        # Denormalize if needed
        if data_for_normalize is not None and data_for_normalize.get('normalize', False):
            U_pred_raw = denormalize_U(U_pred, 
                                        data_for_normalize['U_min'], 
                                        data_for_normalize['U_max'])
            U_test_raw = denormalize_U(U_test.ravel(), 
                                        data_for_normalize['U_min'], 
                                        data_for_normalize['U_max'])
        else:
            U_pred_raw = U_pred
            U_test_raw = U_test.ravel()
        
        metrics = compute_metrics(U_test_raw, U_pred_raw)
        inf_time = compute_inference_time(model, x_test)
        
        row = {
            'Model': name,
            **metrics,
            'Train Time (s)': t_time,
            'Inference (ms)': inf_time,
        }
        rows.append(row)
    
    return pd.DataFrame(rows)


# ============================================================================
# Plotting functions
# ============================================================================

def plot_ocp_comparison(
    x_data: np.ndarray,
    U_data: np.ndarray,
    models: list,
    model_names: list,
    x_plot: np.ndarray,
    electrode_name: str = "Electrode",
    save_path: Optional[str] = None,
    data_for_normalize: Optional[Dict] = None,
):
    """
    Plot OCP data and model predictions.
    
    Parameters
    ----------
    x_data, U_data : np.ndarray
        Full dataset for scatter overlay.
    models : list
        Trained models.
    model_names : list
        Model names for legend.
    x_plot : np.ndarray
        Dense x values for smooth prediction curves.
    electrode_name : str
        Title label.
    save_path : str, optional
        File path to save figure.
    data_for_normalize : dict, optional
        Normalization parameters.
    """
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    
    # Denormalize data if needed
    if data_for_normalize is not None and data_for_normalize.get('normalize', False):
        x_d = denormalize_x(x_data.ravel(), data_for_normalize['x_min'], data_for_normalize['x_max'])
        U_d = denormalize_U(U_data.ravel(), data_for_normalize['U_min'], data_for_normalize['U_max'])
        x_p_raw = denormalize_x(x_plot.ravel(), data_for_normalize['x_min'], data_for_normalize['x_max'])
    else:
        x_d = x_data.ravel()
        U_d = U_data.ravel()
        x_p_raw = x_plot.ravel()
    
    # Data points
    ax.scatter(x_d, U_d, s=8, c='gray', alpha=0.4, label='Literature data', zorder=1)
    
    # Model predictions
    colors = unique_color_cycle(len(models))
    linestyles = ['-', '--', '-.', ':', (0, (5, 1)), (0, (3, 1, 1, 1))]

    for i, (model, name) in enumerate(zip(models, model_names)):
        U_pred = model.predict(x_plot)
        if data_for_normalize is not None and data_for_normalize.get('normalize', False):
            U_pred = denormalize_U(U_pred, data_for_normalize['U_min'], data_for_normalize['U_max'])

        ax.plot(
            x_p_raw,
            U_pred,
            color=colors[i],
            linestyle=linestyles[i % len(linestyles)],
            linewidth=1.8,
            label=name,
            zorder=2 + i,
        )
    
    ax.set_xlabel(r'Stoichiometry $x$')
    ax.set_ylabel(r'OCP $U$ (V vs Li/Li$^+$)')
    ax.set_title(f'{electrode_name} — OCP Model Comparison')
    ax.legend(fontsize=9, loc='best')
    ax.grid(True, alpha=0.3)
    
    if save_path:
        fig.savefig(save_path)
        plt.close(fig)
    else:
        plt.show()


def plot_residuals(
    x_test: np.ndarray,
    U_test: np.ndarray,
    models: list,
    model_names: list,
    electrode_name: str = "Electrode",
    save_path: Optional[str] = None,
    data_for_normalize: Optional[Dict] = None,
):
    """Plot residual errors for each model on the test set."""
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    
    if data_for_normalize is not None and data_for_normalize.get('normalize', False):
        x_raw = denormalize_x(x_test.ravel(), data_for_normalize['x_min'], data_for_normalize['x_max'])
        U_raw = denormalize_U(U_test.ravel(), data_for_normalize['U_min'], data_for_normalize['U_max'])
    else:
        x_raw = x_test.ravel()
        U_raw = U_test.ravel()
    
    colors = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A', '#264653', '#F4A261']
    markers = ['o', 's', '^', 'D', 'v', 'P']
    
    for i, (model, name) in enumerate(zip(models, model_names)):
        U_pred = model.predict(x_test)
        if data_for_normalize is not None and data_for_normalize.get('normalize', False):
            U_pred = denormalize_U(U_pred, data_for_normalize['U_min'], data_for_normalize['U_max'])
        
        residuals = U_raw - U_pred
        ax.scatter(x_raw, residuals * 1000, s=15, alpha=0.6,
                   color=colors[i % len(colors)],
                   marker=markers[i % len(markers)],
                   label=name)
    
    ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel(r'Stoichiometry $x$')
    ax.set_ylabel('Residual (mV)')
    ax.set_title(f'{electrode_name} — Prediction Residuals')
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)
    
    if save_path:
        fig.savefig(save_path)
        plt.close(fig)
    else:
        plt.show()


def plot_derivative_comparison(
    x_plot: np.ndarray,
    pinn_model,
    electrode_name: str = "Electrode",
    true_derivative_func=None,
    save_path: Optional[str] = None,
    data_for_normalize: Optional[Dict] = None,
):
    """
    Plot dU/dx from the PINN and compare with numerical derivative of data.
    
    Parameters
    ----------
    x_plot : np.ndarray
        Stoichiometry values for plotting.
    pinn_model : object with predict_with_derivatives() method
    electrode_name : str
    true_derivative_func : callable, optional
        If provided, compute the true analytical derivative for comparison.
    save_path : str, optional
    """
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    
    U_pred, dU_dx = pinn_model.predict_with_derivatives(x_plot)
    
    if data_for_normalize is not None and data_for_normalize.get('normalize', False):
        x_raw = denormalize_x(x_plot.ravel(), data_for_normalize['x_min'], data_for_normalize['x_max'])
        # Scale derivative: dU/dx_raw = dU_norm/dx_norm * (U_max-U_min)/(x_max-x_min)
        scale = (data_for_normalize['U_max'] - data_for_normalize['U_min']) / \
                (data_for_normalize['x_max'] - data_for_normalize['x_min'] + 1e-10)
        dU_dx_raw = dU_dx * scale
    else:
        x_raw = x_plot.ravel()
        dU_dx_raw = dU_dx
    
    ax.plot(x_raw, dU_dx_raw, 'r-', linewidth=1.5, label='PINN dU/dx')
    
    ax.set_xlabel(r'Stoichiometry $x$')
    ax.set_ylabel(r'$dU/dx$ (V)')
    ax.set_title(f'{electrode_name} — OCP Derivative from PINN')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    if save_path:
        fig.savefig(save_path)
        plt.close(fig)
    else:
        plt.show()


def plot_training_loss(
    train_losses: list,
    val_losses: list,
    model_name: str = "PINN",
    save_path: Optional[str] = None,
):
    """Plot training and validation loss curves."""
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    
    epochs = np.arange(1, len(train_losses) + 1)
    ax.semilogy(epochs, train_losses, 'b-', alpha=0.7, linewidth=0.8, label='Train loss')
    ax.semilogy(epochs, val_losses, 'r-', alpha=0.7, linewidth=0.8, label='Validation loss')
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(f'{model_name} — Training Loss Curve')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    if save_path:
        fig.savefig(save_path)
        plt.close(fig)
    else:
        plt.show()


def plot_free_energy(
    x_plot: np.ndarray,
    fe_pinn_model,
    electrode_name: str = "Electrode",
    save_path: Optional[str] = None,
):
    """Plot the learned free energy profile G(x) from FreeEnergyPINN."""
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    
    G = fe_pinn_model.predict_free_energy(x_plot)
    
    ax.plot(x_plot.ravel(), G, 'b-', linewidth=1.5)
    ax.set_xlabel(r'Stoichiometry $x$')
    ax.set_ylabel(r'$G(x)$ (learned, a.u.)')
    ax.set_title(f'{electrode_name} — Learned Free Energy Profile')
    ax.grid(True, alpha=0.3)
    
    if save_path:
        fig.savefig(save_path)
        plt.close(fig)
    else:
        plt.show()


# ============================================================================
# Main evaluation pipeline
# ============================================================================

def run_full_evaluation(
    electrode_name: str,
    data: Dict,
    pinn_direct,
    pinn_fe,
    baselines: list,
    output_dir: str = "results",
) -> pd.DataFrame:
    """
    Run the complete evaluation pipeline.
    
    Parameters
    ----------
    electrode_name : str
        Name of the electrode for labeling.
    data : dict
        Preprocessed data.
    pinn_direct : DirectOCPPINN
        Trained direct PINN model.
    pinn_fe : FreeEnergyPINN
        Trained free-energy PINN model.
    baselines : list
        Trained baseline models.
    output_dir : str
        Directory for saving figures.
    
    Returns
    -------
    comparison : pd.DataFrame
        Comparison table with metrics for all models.
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    x_test = data['x_test']
    U_test = data['U_test']
    
    # Dense x for plotting
    x_plot = np.linspace(0.0, 1.0, 500).reshape(-1, 1)
    
    # Collect all models
    all_models = [pinn_direct, pinn_fe] + baselines
    all_names = ['PINN (Direct)', 'PINN (Free-Energy)'] + [b.name for b in baselines]
    all_times = [pinn_direct.training_time, pinn_fe.training_time] + \
                [b.training_time for b in baselines]
    
    # Build comparison table
    table = build_comparison_table(
        all_models, all_names, x_test, U_test, all_times,
        data_for_normalize=data
    )
    
    print(f"\n{'='*70}")
    print(f"  {electrode_name} — Model Comparison")
    print(f"{'='*70}")
    print(table.to_string(index=False))
    
    # Save table
    table.to_csv(f"{output_dir}/{electrode_name.lower().replace(' ', '_')}_comparison.csv", 
                 index=False)
    
    # Generate figures
    # 1. OCP comparison
    plot_ocp_comparison(
        data['x_train'], data['U_train'],
        all_models, all_names, x_plot,
        electrode_name=electrode_name,
        save_path=f"{output_dir}/{electrode_name.lower().replace(' ', '_')}_ocp_comparison.png",
        data_for_normalize=data
    )
    
    # 2. Residuals
    plot_residuals(
        x_test, U_test, all_models, all_names,
        electrode_name=electrode_name,
        save_path=f"{output_dir}/{electrode_name.lower().replace(' ', '_')}_residuals.png",
        data_for_normalize=data
    )
    
    # 3. Derivative plot (PINN direct only)
    plot_derivative_comparison(
        x_plot, pinn_direct,
        electrode_name=electrode_name,
        save_path=f"{output_dir}/{electrode_name.lower().replace(' ', '_')}_derivative.png",
        data_for_normalize=data
    )
    
    # 4. Training loss
    plot_training_loss(
        pinn_direct.train_losses, pinn_direct.val_losses,
        model_name=f"{electrode_name} — Direct PINN",
        save_path=f"{output_dir}/{electrode_name.lower().replace(' ', '_')}_training_loss.png"
    )
    
    # 5. Free energy profile
    plot_free_energy(
        x_plot, pinn_fe,
        electrode_name=electrode_name,
        save_path=f"{output_dir}/{electrode_name.lower().replace(' ', '_')}_free_energy.png"
    )
    
    return table


if __name__ == "__main__":
    print("Evaluation module loaded. Use run_full_evaluation() for the full pipeline.")



Overwriting ocp_evaluation.py


In [26]:
from ocp_data_processing_pybamm import generate_ocp_dataset, preprocess_data
from run_ocp_pinn_pybamm import main

# Quick smoke test: create one PyBaMM-backed dataset
df = generate_ocp_dataset('graphite', n_points=20, noise_std=0.0)
df.head()


,x,U_measured,U_true,source,parameter_set,parameter_key,electrode_role
0,0.010000,1.739382,1.739382,PyBaMM bundled parameter set,Chen2020,Negative electrode OCP [V],anode
1,0.061579,0.575121,0.575121,PyBaMM bundled parameter set,Chen2020,Negative electrode OCP [V],anode
2,0.113158,0.362784,0.362784,PyBaMM bundled parameter set,Chen2020,Negative electrode OCP [V],anode
3,0.164737,0.236758,0.236758,PyBaMM bundled parameter set,Chen2020,Negative electrode OCP [V],anode
4,0.216316,0.211058,0.211058,PyBaMM bundled parameter set,Chen2020,Negative electrode OCP [V],anode


In [27]:
%run run_ocp_pinn.py


######################################################################
  Processing: Graphite (Li_xC_6)
######################################################################

[1/5] Generating OCP dataset...
  Data: 200 points, U in [0.092, 1.739] V

[2/5] Preprocessing data...
  Train: 140, Val: 30, Test: 30

[3/5] Training PINNs (5000 epochs)...

  --- Direct OCP PINN ---
Epoch 1000/5000 | Loss: 0.160339 | Data: 0.010156 | Physics: 2.154641 | Val: 0.004645
Epoch 2000/5000 | Loss: 0.143571 | Data: 0.011057 | Physics: 1.363295 | Val: 0.005097
Epoch 3000/5000 | Loss: 0.073542 | Data: 0.010599 | Physics: 0.501523 | Val: 0.004728
Epoch 4000/5000 | Loss: 0.121584 | Data: 0.010665 | Physics: 1.067730 | Val: 0.004732
Epoch 5000/5000 | Loss: 0.096316 | Data: 0.010701 | Physics: 0.805866 | Val: 0.004768
  Direct PINN training time: 164.23 s

  --- Free-Energy PINN ---
Epoch 1000/5000 | Loss: 0.000068 | Data: 0.000068 | Convex: 0.000000 | Val: 0.000059
Epoch 2000/5000 | Loss: 0.000032 | Data: 